---
## 1. Imports & Configuration

In [1]:
import os
import math
import random
from collections import Counter, defaultdict

import pyarrow.parquet as pq
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────
PARQUET_DIR = os.path.join("..", "Assignment_1", "data", "output")
FIRST_SHARD = os.path.join(PARQUET_DIR, "indiccorp_te_tokenized_part-00001.parquet")

# ── Parameters ─────────────────────────────────────────────────────────────
NUM_SENTENCES = 1_000_000   # at least 1M sentences required
DEV_SIZE      = 1_000
TEST_SIZE     = 1_000
TRAIN_SIZE    = NUM_SENTENCES - DEV_SIZE - TEST_SIZE   # 998,000
SEED          = 42          # reproducibility

# Special tokens for sentence boundaries
SOS = "<s>"    # start of sentence
EOS = "</s>"   # end of sentence
UNK = "<UNK>"  # unknown token (used during evaluation)

print(f"Will load {NUM_SENTENCES:,} sentences from:\n  {os.path.abspath(FIRST_SHARD)}")
print(f"Split: train={TRAIN_SIZE:,}  dev={DEV_SIZE:,}  test={TEST_SIZE:,}")

Will load 1,000,000 sentences from:
  d:\Assignments_sem_5\NLP\Assignment_1\data\output\indiccorp_te_tokenized_part-00001.parquet
Split: train=998,000  dev=1,000  test=1,000


---
## 2. Load Tokenized Data

In [2]:
# Read the first parquet shard (contains exactly 1,000,000 tokenized sentences)
table = pq.read_table(FIRST_SHARD, columns=["tokenized_sentence"])
raw_sentences = table.column("tokenized_sentence").to_pylist()

print(f"Loaded {len(raw_sentences):,} sentences from the first shard.")
print(f"\nSample sentences:")
for i in range(5):
    print(f"  [{i}] {raw_sentences[i][:120]}{'…' if len(raw_sentences[i]) > 120 else ''}")

Loaded 1,000,000 sentences from the first shard.

Sample sentences:
  [0] అమెరికా అధ్యక్షుడు డొనాల్డ్ ట్రంప్ కు రాష్ట్రపతి భవన్ వద్ద ఘనస్వాగతం లభించింది .
  [1] ఆయనకు రాష్ట్రపతి రామ్ నాథ్ కోవింద్ దంపతులు , ప్రధాని మోదీ సాదరంగా ఆహ్వానం పలకడంతో పాటు సైనికులు గౌరవ వందనాన్ని అందించారు…
  [2] ఇటు తెలంగాణలో కరోనా వైరస్ కారణంగా అన్ని దేవాలయాల్లో ముందస్తు చర్యలు చేపట్టారు .
  [3] భద్రాద్రి రాముడికి కరోనా ఎఫెక్ట్ తగిలింది .
  [4] ఏప్రిల్ 2న భద్రాద్రిలో జరగనున్న శ్రీరామ నవమి వేడుకలను వెంటాడుతోంది కరోనా .


In [3]:
# Convert each sentence string to a list of tokens
# Each tokenized_sentence is space-separated tokens from Assignment 1
sentences = [s.split() for s in raw_sentences[:NUM_SENTENCES]]

print(f"Total sentences: {len(sentences):,}")
print(f"Sample tokenized sentence: {sentences[0][:15]}{'…' if len(sentences[0]) > 15 else ''}")

Total sentences: 1,000,000
Sample tokenized sentence: ['అమెరికా', 'అధ్యక్షుడు', 'డొనాల్డ్', 'ట్రంప్', 'కు', 'రాష్ట్రపతి', 'భవన్', 'వద్ద', 'ఘనస్వాగతం', 'లభించింది', '.']


---
## 3. Train / Development / Test Split

In [4]:
# Shuffle for randomness, then split
random.seed(SEED)
random.shuffle(sentences)

train_sentences = sentences[:TRAIN_SIZE]
dev_sentences   = sentences[TRAIN_SIZE : TRAIN_SIZE + DEV_SIZE]
test_sentences  = sentences[TRAIN_SIZE + DEV_SIZE : TRAIN_SIZE + DEV_SIZE + TEST_SIZE]

print(f"Training set  : {len(train_sentences):>10,} sentences")
print(f"Development set: {len(dev_sentences):>10,} sentences")
print(f"Test set       : {len(test_sentences):>10,} sentences")

# Free memory
del raw_sentences, table, sentences

Training set  :    998,000 sentences
Development set:      1,000 sentences
Test set       :      1,000 sentences


---
## 4. Build Vocabulary from Training Data

In [5]:
# Build vocabulary from training data only
vocab = set()
total_train_tokens = 0

for sent in tqdm(train_sentences, desc="Building vocabulary"):
    vocab.update(sent)
    total_train_tokens += len(sent)

# Add special tokens to vocab
vocab.add(SOS)
vocab.add(EOS)
vocab.add(UNK)

V = len(vocab)  # vocabulary size (including special tokens)

print(f"Vocabulary size (V): {V:,}")
print(f"Total training tokens (before adding boundary markers): {total_train_tokens:,}")

Building vocabulary: 100%|██████████| 998000/998000 [00:02<00:00, 448768.96it/s]

Vocabulary size (V): 670,753
Total training tokens (before adding boundary markers): 10,441,087


In [6]:
def pad_sentence(tokens, n):
    """
    Pad a sentence with (n-1) <s> tokens at the start and one </s> at the end.
    
    For bigram  (n=2): [<s>, w1, w2, ..., wk, </s>]
    For trigram (n=3): [<s>, <s>, w1, w2, ..., wk, </s>]
    For quadrigram (n=4): [<s>, <s>, <s>, w1, w2, ..., wk, </s>]
    """
    return [SOS] * (n - 1) + tokens + [EOS]


def get_ngrams(tokens, n):
    """Extract all n-grams from a token list."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


# Quick demo
demo = ["నమస్తే", "తెలంగాణ", "ప్రత్యేక"]
for n in [1, 2, 3, 4]:
    padded = pad_sentence(demo, n)
    ngrams = get_ngrams(padded, n)
    print(f"n={n}: padded={padded}")
    print(f"       n-grams={ngrams}\n")

n=1: padded=['నమస్తే', 'తెలంగాణ', 'ప్రత్యేక', '</s>']
       n-grams=[('నమస్తే',), ('తెలంగాణ',), ('ప్రత్యేక',), ('</s>',)]

n=2: padded=['<s>', 'నమస్తే', 'తెలంగాణ', 'ప్రత్యేక', '</s>']
       n-grams=[('<s>', 'నమస్తే'), ('నమస్తే', 'తెలంగాణ'), ('తెలంగాణ', 'ప్రత్యేక'), ('ప్రత్యేక', '</s>')]

n=3: padded=['<s>', '<s>', 'నమస్తే', 'తెలంగాణ', 'ప్రత్యేక', '</s>']
       n-grams=[('<s>', '<s>', 'నమస్తే'), ('<s>', 'నమస్తే', 'తెలంగాణ'), ('నమస్తే', 'తెలంగాణ', 'ప్రత్యేక'), ('తెలంగాణ', 'ప్రత్యేక', '</s>')]

n=4: padded=['<s>', '<s>', '<s>', 'నమస్తే', 'తెలంగాణ', 'ప్రత్యేక', '</s>']
       n-grams=[('<s>', '<s>', '<s>', 'నమస్తే'), ('<s>', '<s>', 'నమస్తే', 'తెలంగాణ'), ('<s>', 'నమస్తే', 'తెలంగాణ', 'ప్రత్యేక'), ('నమస్తే', 'తెలంగాణ', 'ప్రత్యేక', '</s>')]



---
## 6. Build N-gram Counts from Training Data

In [7]:
# ── Count n-grams for n = 1, 2, 3, 4 ─────────────────────────────────────

# Unigram counts (no padding needed, but we add </s> for consistency)
unigram_counts = Counter()

# Bigram counts & context (unigram) counts
bigram_counts  = Counter()
bigram_context = Counter()   # counts of (w_{i-1},)

# Trigram counts & context (bigram) counts
trigram_counts  = Counter()
trigram_context = Counter()  # counts of (w_{i-2}, w_{i-1})

# Quadrigram counts & context (trigram) counts
quadrigram_counts  = Counter()
quadrigram_context = Counter()  # counts of (w_{i-3}, w_{i-2}, w_{i-1})

for sent in tqdm(train_sentences, desc="Counting n-grams"):
    # --- Unigrams (pad with 0 start tokens + 1 end token) ---
    padded_1 = pad_sentence(sent, 1)  # just adds </s>
    for gram in get_ngrams(padded_1, 1):
        unigram_counts[gram] += 1
    
    # --- Bigrams ---
    padded_2 = pad_sentence(sent, 2)
    for gram in get_ngrams(padded_2, 2):
        bigram_counts[gram] += 1
        bigram_context[gram[:1]] += 1
    
    # --- Trigrams ---
    padded_3 = pad_sentence(sent, 3)
    for gram in get_ngrams(padded_3, 3):
        trigram_counts[gram] += 1
        trigram_context[gram[:2]] += 1
    
    # --- Quadrigrams ---
    padded_4 = pad_sentence(sent, 4)
    for gram in get_ngrams(padded_4, 4):
        quadrigram_counts[gram] += 1
        quadrigram_context[gram[:3]] += 1

N_unigrams = sum(unigram_counts.values())

print(f"\n{'Model':<12} {'Unique n-grams':>16} {'Total count':>14}")
print("-" * 46)
print(f"{'Unigram':<12} {len(unigram_counts):>16,} {N_unigrams:>14,}")
print(f"{'Bigram':<12} {len(bigram_counts):>16,} {sum(bigram_counts.values()):>14,}")
print(f"{'Trigram':<12} {len(trigram_counts):>16,} {sum(trigram_counts.values()):>14,}")
print(f"{'Quadrigram':<12} {len(quadrigram_counts):>16,} {sum(quadrigram_counts.values()):>14,}")

Counting n-grams: 100%|██████████| 998000/998000 [01:06<00:00, 14969.99it/s]



Model          Unique n-grams    Total count
----------------------------------------------
Unigram               670,751     11,439,087
Bigram              4,797,059     11,439,087
Trigram             7,831,299     11,439,087
Quadrigram          9,045,911     11,439,087


In [8]:
def unigram_prob(word):
    """P(word) = count(word) / N"""
    return unigram_counts.get((word,), 0) / N_unigrams


def sentence_prob_unigram(tokens):
    """Compute log probability of a sentence under the unigram model."""
    padded = pad_sentence(tokens, 1)
    log_prob = 0.0
    n_tokens = 0
    for w in padded:
        p = unigram_prob(w)
        if p > 0:
            log_prob += math.log2(p)
        else:
            log_prob += float('-inf')  # unseen token
        n_tokens += 1
    return log_prob, n_tokens


# ── Display top-20 unigrams ──────────────────────────────────────────────
print("Top-20 most frequent unigrams:\n")
print(f"{'Rank':<6} {'Token':<20} {'Count':>12} {'P(token)':>12}")
print("-" * 54)
for rank, (gram, count) in enumerate(unigram_counts.most_common(20), 1):
    token = gram[0]
    prob  = count / N_unigrams
    print(f"{rank:<6} {token:<20} {count:>12,} {prob:>12.6f}")

# ── Sample sentence probability ──────────────────────────────────────────
sample = train_sentences[0]
lp, nt = sentence_prob_unigram(sample)
print(f"\nSample sentence: {' '.join(sample[:10])}{'…' if len(sample) > 10 else ''}")
print(f"Log₂ P(sentence) = {lp:.4f}")
print(f"Tokens = {nt}")

Top-20 most frequent unigrams:

Rank   Token                       Count     P(token)
------------------------------------------------------
1      </s>                      998,000     0.087245
2      .                         903,196     0.078957
3      ,                         354,860     0.031022
4      ఈ                         112,757     0.009857
5      కూడా                       50,538     0.004418
6      ఆ                          36,817     0.003219
7      :                          31,561     0.002759
8      లో                         30,672     0.002681
9      )                          30,349     0.002653
10     ఒక                         30,345     0.002653
11     నుంచి                      30,000     0.002623
12     (                          29,987     0.002621
13     ఉంది                       29,376     0.002568
14     -                          26,181     0.002289
15     మరియు                      26,140     0.002285
16     నుండి                      25,274     0.00

In [9]:
def bigram_prob(w_prev, w_curr):
    """P(w_curr | w_prev) = count(w_prev, w_curr) / count(w_prev)"""
    numerator   = bigram_counts.get((w_prev, w_curr), 0)
    denominator = bigram_context.get((w_prev,), 0)
    if denominator == 0:
        return 0.0
    return numerator / denominator


def sentence_prob_bigram(tokens):
    """Compute log probability of a sentence under the bigram model."""
    padded = pad_sentence(tokens, 2)
    log_prob = 0.0
    n_tokens = 0
    for i in range(1, len(padded)):
        p = bigram_prob(padded[i-1], padded[i])
        if p > 0:
            log_prob += math.log2(p)
        else:
            log_prob += float('-inf')
        n_tokens += 1
    return log_prob, n_tokens


# ── Display top-20 bigrams ───────────────────────────────────────────────
print("Top-20 most frequent bigrams:\n")
print(f"{'Rank':<6} {'Bigram':<35} {'Count':>12} {'P(w₂|w₁)':>12}")
print("-" * 69)
for rank, (gram, count) in enumerate(bigram_counts.most_common(20), 1):
    prob = bigram_prob(gram[0], gram[1])
    label = f"{gram[0]} → {gram[1]}"
    print(f"{rank:<6} {label:<35} {count:>12,} {prob:>12.6f}")

# ── Sample sentence probability ──────────────────────────────────────────
lp, nt = sentence_prob_bigram(sample)
print(f"\nSample sentence: {' '.join(sample[:10])}{'…' if len(sample) > 10 else ''}")
print(f"Log₂ P(sentence) = {lp:.4f}")
print(f"Bigram tokens = {nt}")

Top-20 most frequent bigrams:

Rank   Bigram                                     Count     P(w₂|w₁)
---------------------------------------------------------------------
1      . → </s>                                 895,605     0.991595
2      <s> → ఈ                                   51,837     0.051941
3      ఉంది → .                                  25,463     0.866796
4      చేశారు → .                                21,409     0.954310
5      ? → </s>                                  19,327     0.913245
6      ఉన్నాయి → .                               19,264     0.944592
7      <s> → అయితే                               18,891     0.018929
8      <s> → ఆ                                   14,500     0.014529
9      రూ → .                                    14,082     0.980504
10     ! → </s>                                  12,526     0.926479
11     తెలిపారు → .                              11,826     0.982716
12     <s> → దీంతో                               10,449     0.010470
13

In [10]:
def trigram_prob(w_2, w_1, w_0):
    """P(w_0 | w_2, w_1) = count(w_2, w_1, w_0) / count(w_2, w_1)"""
    numerator   = trigram_counts.get((w_2, w_1, w_0), 0)
    denominator = trigram_context.get((w_2, w_1), 0)
    if denominator == 0:
        return 0.0
    return numerator / denominator


def sentence_prob_trigram(tokens):
    """Compute log probability of a sentence under the trigram model."""
    padded = pad_sentence(tokens, 3)
    log_prob = 0.0
    n_tokens = 0
    for i in range(2, len(padded)):
        p = trigram_prob(padded[i-2], padded[i-1], padded[i])
        if p > 0:
            log_prob += math.log2(p)
        else:
            log_prob += float('-inf')
        n_tokens += 1
    return log_prob, n_tokens


# ── Display top-20 trigrams ──────────────────────────────────────────────
print("Top-20 most frequent trigrams:\n")
print(f"{'Rank':<6} {'Trigram':<50} {'Count':>10}")
print("-" * 70)
for rank, (gram, count) in enumerate(trigram_counts.most_common(20), 1):
    label = f"{gram[0]} → {gram[1]} → {gram[2]}"
    print(f"{rank:<6} {label:<50} {count:>10,}")

# ── Sample sentence probability ──────────────────────────────────────────
lp, nt = sentence_prob_trigram(sample)
print(f"\nSample sentence: {' '.join(sample[:10])}{'…' if len(sample) > 10 else ''}")
print(f"Log₂ P(sentence) = {lp:.4f}")
print(f"Trigram tokens = {nt}")

Top-20 most frequent trigrams:

Rank   Trigram                                                 Count
----------------------------------------------------------------------
1      <s> → <s> → ఈ                                          51,837
2      ఉంది → . → </s>                                        25,420
3      చేశారు → . → </s>                                      21,390
4      ఉన్నాయి → . → </s>                                     19,248
5      <s> → <s> → అయితే                                      18,891
6      <s> → <s> → ఆ                                          14,500
7      రూ → . → </s>                                          14,082
8      తెలిపారు → . → </s>                                    11,822
9      <s> → <s> → దీంతో                                      10,449
10     అన్నారు → . → </s>                                     10,348
11     <s> → <s> → కానీ                                       10,225
12     <s> → <s> → ఇక                                          9,141


In [11]:
def quadrigram_prob(w_3, w_2, w_1, w_0):
    """P(w_0 | w_3, w_2, w_1) = count(w_3, w_2, w_1, w_0) / count(w_3, w_2, w_1)"""
    numerator   = quadrigram_counts.get((w_3, w_2, w_1, w_0), 0)
    denominator = quadrigram_context.get((w_3, w_2, w_1), 0)
    if denominator == 0:
        return 0.0
    return numerator / denominator


def sentence_prob_quadrigram(tokens):
    """Compute log probability of a sentence under the quadrigram model."""
    padded = pad_sentence(tokens, 4)
    log_prob = 0.0
    n_tokens = 0
    for i in range(3, len(padded)):
        p = quadrigram_prob(padded[i-3], padded[i-2], padded[i-1], padded[i])
        if p > 0:
            log_prob += math.log2(p)
        else:
            log_prob += float('-inf')
        n_tokens += 1
    return log_prob, n_tokens


# ── Display top-20 quadrigrams ───────────────────────────────────────────
print("Top-20 most frequent quadrigrams:\n")
print(f"{'Rank':<6} {'Quadrigram':<65} {'Count':>10}")
print("-" * 85)
for rank, (gram, count) in enumerate(quadrigram_counts.most_common(20), 1):
    label = f"{gram[0]} → {gram[1]} → {gram[2]} → {gram[3]}"
    print(f"{rank:<6} {label:<65} {count:>10,}")

# ── Sample sentence probability ──────────────────────────────────────────
lp, nt = sentence_prob_quadrigram(sample)
print(f"\nSample sentence: {' '.join(sample[:10])}{'…' if len(sample) > 10 else ''}")
print(f"Log₂ P(sentence) = {lp:.4f}")
print(f"Quadrigram tokens = {nt}")

Top-20 most frequent quadrigrams:

Rank   Quadrigram                                                             Count
-------------------------------------------------------------------------------------
1      <s> → <s> → <s> → ఈ                                                   51,837
2      <s> → <s> → <s> → అయితే                                               18,891
3      <s> → <s> → <s> → ఆ                                                   14,500
4      <s> → <s> → <s> → దీంతో                                               10,449
5      <s> → <s> → <s> → కానీ                                                10,225
6      <s> → <s> → <s> → ఇక                                                   9,141
7      <s> → <s> → <s> → ఇది                                                  8,233
8      <s> → <s> → <s> → ‘                                                    7,745
9      <s> → <s> → <s> → గ్రామంలో                                             6,269
10     దూరంలో → ఉన్నాయి → . → </s>     

In [12]:
def compute_perplexity(data_sentences, sentence_prob_fn):
    """
    Compute perplexity of a dataset under a given language model.
    
    Parameters
    ----------
    data_sentences : list of list of str
        The evaluation sentences (each is a list of tokens).
    sentence_prob_fn : callable
        Function that takes a token list and returns (log2_prob, n_tokens).
    
    Returns
    -------
    float
        The perplexity. Returns float('inf') if any token has zero probability.
    """
    total_log_prob = 0.0
    total_tokens   = 0
    
    for sent in data_sentences:
        lp, nt = sentence_prob_fn(sent)
        if math.isinf(lp):
            return float('inf')   # unseen n-gram encountered
        total_log_prob += lp
        total_tokens   += nt
    
    avg_log_prob = total_log_prob / total_tokens
    perplexity   = 2 ** (-avg_log_prob)
    return perplexity


print("Computing perplexity on development and test sets...\n")

Computing perplexity on development and test sets...



In [13]:
# ── Compute perplexity for all 4 models on dev and test sets ──────────────
models = {
    "Unigram":    sentence_prob_unigram,
    "Bigram":     sentence_prob_bigram,
    "Trigram":    sentence_prob_trigram,
    "Quadrigram": sentence_prob_quadrigram,
}

results = {}

for name, prob_fn in models.items():
    pp_dev  = compute_perplexity(dev_sentences, prob_fn)
    pp_test = compute_perplexity(test_sentences, prob_fn)
    results[name] = {"dev": pp_dev, "test": pp_test}
    print(f"{name:<12}  Dev PP = {pp_dev:<20}  Test PP = {pp_test}")

print("\n(inf = model encountered unseen n-grams; smoothing is needed — see Question 2)")

Unigram       Dev PP = inf                   Test PP = inf
Bigram        Dev PP = inf                   Test PP = inf
Trigram       Dev PP = inf                   Test PP = inf
Quadrigram    Dev PP = inf                   Test PP = inf

(inf = model encountered unseen n-grams; smoothing is needed — see Question 2)


---
## 12. Summary

In [14]:
print("=" * 80)
print("SUMMARY — N-gram Language Models (No Smoothing)")
print("=" * 80)
print(f"\nCorpus           : IndicCorpV2 Telugu (tel_Telu)")
print(f"Total sentences  : {NUM_SENTENCES:,}")
print(f"Training set     : {TRAIN_SIZE:,} sentences")
print(f"Development set  : {DEV_SIZE:,} sentences")
print(f"Test set         : {TEST_SIZE:,} sentences")
print(f"Vocabulary size  : {V:,}")
print()

print(f"{'Model':<12} {'Unique n-grams':>16} {'Dev PP':>20} {'Test PP':>20}")
print("-" * 72)
ngram_data = [
    ("Unigram",    len(unigram_counts)),
    ("Bigram",     len(bigram_counts)),
    ("Trigram",    len(trigram_counts)),
    ("Quadrigram", len(quadrigram_counts)),
]
for name, unique_count in ngram_data:
    dev_pp  = results[name]["dev"]
    test_pp = results[name]["test"]
    dev_str  = f"{dev_pp:,.2f}" if not math.isinf(dev_pp) else "∞"
    test_str = f"{test_pp:,.2f}" if not math.isinf(test_pp) else "∞"
    print(f"{name:<12} {unique_count:>16,} {dev_str:>20} {test_str:>20}")

print("\n" + "=" * 80)
print("Note: Infinite perplexity is expected without smoothing when unseen n-grams")
print("appear in dev/test data. See Question 2 for Laplace (Add-1) Smoothing.")
print("=" * 80)

SUMMARY — N-gram Language Models (No Smoothing)

Corpus           : IndicCorpV2 Telugu (tel_Telu)
Total sentences  : 1,000,000
Training set     : 998,000 sentences
Development set  : 1,000 sentences
Test set         : 1,000 sentences
Vocabulary size  : 670,753

Model          Unique n-grams               Dev PP              Test PP
------------------------------------------------------------------------
Unigram               670,751                    ∞                    ∞
Bigram              4,797,059                    ∞                    ∞
Trigram             7,831,299                    ∞                    ∞
Quadrigram          9,045,911                    ∞                    ∞

Note: Infinite perplexity is expected without smoothing when unseen n-grams
appear in dev/test data. See Question 2 for Laplace (Add-1) Smoothing.
